# 🧠 Neural Network from Scratch — Manual Backpropagation

**Dataset:** FashionMNIST — 60,000 grayscale images, 10 clothing classes  
**Architecture:** MLP — 784 → 128 → 10  
**Training:** Full manual backpropagation, no autograd, pure NumPy  

## Architecture

| Layer | Operation | Shape |
|-------|-----------|-------|
| Input | Raw pixels flattened | (N, 784) |
| Hidden | Z1 = X @ W1 + b1 | (N, 128) |
| Hidden Act | A1 = ReLU(Z1) | (N, 128) |
| Output | Z2 = A1 @ W2 + b2 | (N, 10) |
| Output Act | A2 = Softmax(Z2) | (N, 10) |

## Loss — Categorical Cross-Entropy + L2

$$\mathcal{L} = -\frac{1}{N} \sum_{i} \sum_{c} y_{ic} \log(\hat{y}_{ic} + \varepsilon) + \frac{\lambda}{2N}(\|W_1\|^2 + \|W_2\|^2)$$

## Backpropagation — Chain Rule

**Layer 2:**
$$dZ_2 = A_2 - y \quad \text{(combined Softmax + CCE derivative)}$$
$$dW_2 = \frac{1}{N} A_1^T \cdot dZ_2 + \frac{\lambda}{N} W_2$$

**Layer 1 (through ReLU):**
$$dA_1 = dZ_2 \cdot W_2^T$$
$$dZ_1 = dA_1 \odot \text{ReLU}'(Z_1) \quad \text{(element-wise gate)}$$
$$dW_1 = \frac{1}{N} X^T \cdot dZ_1 + \frac{\lambda}{N} W_1$$

In [ ]:
# ── Imports & Config ──────────────────────────────────────────────────────────
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import yaml
from torchvision import datasets
from src.neural_network import NeuralNetwork

with open("../configs/nn_config.yaml", "r") as f:
    config = yaml.safe_load(f)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",      "Shirt",   "Sneaker",  "Bag",  "Ankle boot"
]

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")

In [ ]:
# ── Load & Prepare Data ───────────────────────────────────────────────────────
train_data = datasets.FashionMNIST(root="../data", train=True, download=True)

# .data  → torch.Tensor (60000, 28, 28) uint8
# .numpy() converts to np.ndarray — needed for NumPy operations
X = train_data.data.numpy()      # (60000, 28, 28)
y = train_data.targets.numpy()   # (60000,)

# Flatten: 28x28 → 784
# Why: our Linear layer expects a 1D feature vector per sample.
# -1 tells NumPy to infer this dimension (= 60000 here).
X = X.reshape(-1, 784)           # (60000, 784)

# Normalize: uint8 [0,255] → float32 [0,1]
# Why: keeps weight gradients numerically stable.
# Without normalization, pixel values of 200+ cause large Z1 activations
# which push softmax outputs to extreme values near 0 or 1.
X = X / 255.0                    # (60000, 784)

# Split
split   = config["data"]["val_split"]     # 55000
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"Pixel range : [{X_train.min():.1f}, {X_train.max():.1f}]")

In [ ]:
# ── Train Model ───────────────────────────────────────────────────────────────
model = NeuralNetwork(config)
model.fit(X_train, y_train, X_val, y_val)

print(f"\nFinal train loss : {model.loss_history[-1]:.6f}")
print(f"Final train acc  : {model.accuracy_history[-1]:.4f}")
print(f"W1 shape         : {model.W1.shape}")
print(f"W2 shape         : {model.W2.shape}")

In [ ]:
# ── Loss & Accuracy Curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(model.loss_history, color="#1A6BC4", linewidth=1.8)
axes[0].set_xlabel("Epoch");  axes[0].set_ylabel("CCE Loss")
axes[0].set_title("Training Loss Curve"); axes[0].grid(alpha=0.3)

axes[1].plot(model.accuracy_history, color="#00C28B", linewidth=1.8)
axes[1].set_xlabel("Epoch");  axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training Accuracy Curve"); axes[1].grid(alpha=0.3)

plt.tight_layout()
os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/loss_accuracy_curve_nn.png", dpi=150)
plt.show()
print("Saved → assets/loss_accuracy_curve_nn.png")

In [ ]:
# ── Evaluate on Validation Set ────────────────────────────────────────────────
metrics = model.evaluate(X_val, y_val)

print("─" * 38)
print("  Evaluation — Validation Set")
print("─" * 38)
for k, v in metrics.items():
    print(f"  {k:<18} : {v}")
print("─" * 38)

In [ ]:
# ── Per-Class Accuracy ────────────────────────────────────────────────────────
# Which clothing items does the model find hardest to classify?
# Shirt vs T-shirt/top vs Pullover often get confused.
y_pred      = model.predict(X_val)
per_class   = {}
for c in range(10):
    mask           = (y_val == c)
    per_class[c]   = float(np.mean(y_pred[mask] == y_val[mask]))

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(CLASS_NAMES, [per_class[c] for c in range(10)],
              color="#1A6BC4", alpha=0.85, edgecolor="white")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Per-Class Validation Accuracy — FashionMNIST", fontsize=14)
ax.tick_params(axis="x", rotation=30)
for bar, val in zip(bars, per_class.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../assets/per_class_accuracy_nn.png", dpi=150)
plt.show()
print("Saved → assets/per_class_accuracy_nn.png")

In [ ]:
# ── Sample Predictions Grid ───────────────────────────────────────────────────
# Visual check: does the model get easy cases right?
# Green title = correct prediction. Red = wrong.
rng      = np.random.default_rng(42)
sample_idx = rng.choice(len(X_val), size=16, replace=False)
sample_X   = X_val[sample_idx]
sample_y   = y_val[sample_idx]
sample_pred = model.predict(sample_X)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(sample_X[i].reshape(28, 28), cmap="gray")
    correct = sample_pred[i] == sample_y[i]
    color   = "#2ECC71" if correct else "#E74C3C"
    ax.set_title(f"P: {CLASS_NAMES[sample_pred[i]]}\nT: {CLASS_NAMES[sample_y[i]]}",
                 fontsize=7, color=color)
    ax.axis("off")
plt.suptitle("Sample Predictions (Green=Correct, Red=Wrong)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../assets/sample_predictions_nn.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → assets/sample_predictions_nn.png")

## Results Summary

| Metric | Value |
|--------|-------|
| Val Accuracy | *see output above* |
| Val Loss     | *see output above* |

**Key Takeaways:**
- **He initialization** prevents vanishing gradients with ReLU activations  
- **ReLU derivative** gates gradient flow — dead neurons (Z ≤ 0) receive zero gradient  
- **dZ2 = A2 - y** is the elegant combined derivative of Softmax + Cross-Entropy  
- **Shirt, Pullover, T-shirt** are hardest to separate — visually similar textures  
- **Trouser, Bag, Ankle Boot** are easiest — very distinct shapes